# Check Embedding Sizes & Insurance Labels: NPZ vs Pickle (All Strategies)

Validates embeddings and insurance labels for **all 11 sampling strategies**:

| # | Strategy | Type | Expected Samples |
|---|----------|------|------------------|
| 1 | Balanced Insurance (50/50) | Random | ~2,000 |
| 2 | Stratified Proportional | Random | ~2,000 |
| 3 | Simple Random | Random | ~2,000 |
| 4 | Pathology Stratified | Random | ~2,000 |
| 5 | Pathology Stratified Clean | Random | ~2,000 |
| 6 | ContextualDiversity | Coreset | ~1,100-1,185 |
| 7 | Forgetting | Coreset | ~1,100-1,185 |
| 8 | Glister | Coreset | ~1,100-1,185 |
| 9 | GradMatch | Coreset | ~1,100-1,185 |
| 10 | Submodular | Coreset | ~1,100-1,185 |
| 11 | Uncertainty | Coreset | ~1,100-1,185 |

**Insurance Labels:**
- Private → Class 1 ([0., 1.])
- Medicaid/Medicare → Class 0 ([1., 0.])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import os
from glob import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

CONFIG = {
    'EMBEDDING_PATH': "/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/ve_cxr/elixir/batch_*.npz",
    'DATA_DIR': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned',
    'OUTPUT_DIR': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned',
    'INSURANCE_COLUMN': 'new_insurance_type',
    'VERBOSE': True
}

STRATEGIES = {
    1: {'name': 'Balanced (50/50)', 'type': 'Random'},
    2: {'name': 'Stratified Proportional', 'type': 'Random'},
    3: {'name': 'Simple Random', 'type': 'Random'},
    4: {'name': 'Pathology Stratified', 'type': 'Random'},
    5: {'name': 'Pathology Stratified Clean', 'type': 'Random'},
    6: {'name': 'ContextualDiversity', 'type': 'Coreset'},
    7: {'name': 'Forgetting', 'type': 'Coreset'},
    8: {'name': 'Glister', 'type': 'Coreset'},
    9: {'name': 'GradMatch', 'type': 'Coreset'},
    10: {'name': 'Submodular', 'type': 'Coreset'},
    11: {'name': 'Uncertainty', 'type': 'Coreset'}
}

os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)
print("Configuration loaded.")
print(f"Data directory: {CONFIG['DATA_DIR']}")

## 1. Check NPZ Reference

In [ ]:
print("=" * 80)
print("1. CHECKING NPZ REFERENCE")
print("=" * 80)

batch_files = sorted(glob(CONFIG['EMBEDDING_PATH']))
print(f"Found {len(batch_files)} batch files")

npz_shape = None
npz_flat_size = None

if batch_files:
    data = np.load(batch_files[0], allow_pickle=True)
    first_emb = data['embeddings'][0]
    npz_shape = first_emb.shape
    npz_flat_size = int(np.prod(npz_shape))
    
    print(f"\n[NPZ REFERENCE]")
    print(f"  Shape: {npz_shape}")
    print(f"  Flattened: {npz_flat_size:,} features")
    print(f"  Dtype: {first_emb.dtype}")
else:
    print("\nERROR: No NPZ files found!")

## 2. Check All Strategy Pickle Files

In [ ]:
print("\n" + "=" * 80)
print("2. CHECKING ALL STRATEGY PICKLE FILES")
print("=" * 80)

results = []

for strategy_num in range(1, 12):
    pickle_file = os.path.join(CONFIG['DATA_DIR'], f"data_type{strategy_num}_insurance.pkl")
    
    print(f"\n[Strategy {strategy_num}: {STRATEGIES[strategy_num]['name']}]")
    
    if not os.path.exists(pickle_file):
        print(f"  ✗ File not found: {os.path.basename(pickle_file)}")
        results.append({
            'strategy': strategy_num,
            'name': STRATEGIES[strategy_num]['name'],
            'type': STRATEGIES[strategy_num]['type'],
            'status': 'Not found',
            'samples': 0,
            'embedding_shape': 'N/A',
            'embedding_size': 0,
            'private': 0,
            'medicaid_medicare': 0,
            'private_pct': 0.0,
            'file_size_mb': 0.0
        })
        continue
    
    try:
        df = pd.read_pickle(pickle_file)
        
        # Embedding info
        emb_shape = df['embedding'].iloc[0].shape
        emb_size = int(np.prod(emb_shape))
        
        # Insurance distribution
        ins_counts = df[CONFIG['INSURANCE_COLUMN']].value_counts()
        private_count = ins_counts.get('Private', 0)
        medicaid_count = ins_counts.get('Medicaid_Medicare', 0)
        private_pct = (private_count / len(df)) * 100
        
        # File size
        file_size = os.path.getsize(pickle_file) / (1024 * 1024)
        
        # Status
        status = '✓ OK' if emb_size == npz_flat_size else '✗ Mismatch'
        
        print(f"  {status}")
        print(f"  Samples: {len(df):,}")
        print(f"  Embedding: {emb_shape} → {emb_size:,} features")
        print(f"  Insurance: Private={private_count:,} ({private_pct:.1f}%), Medicaid/Medicare={medicaid_count:,} ({100-private_pct:.1f}%)")
        print(f"  File size: {file_size:.2f} MB")
        
        results.append({
            'strategy': strategy_num,
            'name': STRATEGIES[strategy_num]['name'],
            'type': STRATEGIES[strategy_num]['type'],
            'status': status,
            'samples': len(df),
            'embedding_shape': str(emb_shape),
            'embedding_size': emb_size,
            'private': private_count,
            'medicaid_medicare': medicaid_count,
            'private_pct': private_pct,
            'file_size_mb': file_size
        })
        
    except Exception as e:
        print(f"  ✗ Error loading file: {e}")
        results.append({
            'strategy': strategy_num,
            'name': STRATEGIES[strategy_num]['name'],
            'type': STRATEGIES[strategy_num]['type'],
            'status': 'Error',
            'samples': 0,
            'embedding_shape': 'N/A',
            'embedding_size': 0,
            'private': 0,
            'medicaid_medicare': 0,
            'private_pct': 0.0,
            'file_size_mb': 0.0
        })

results_df = pd.DataFrame(results)
print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)
print(results_df.to_string(index=False))

## 3. Comparison Visualization

In [ ]:
# Filter valid results
valid_results = results_df[results_df['samples'] > 0].copy()

if len(valid_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Sample counts by strategy
    ax1 = axes[0, 0]
    colors = ['#3498db' if t == 'Random' else '#e74c3c' for t in valid_results['type']]
    bars = ax1.bar(valid_results['strategy'], valid_results['samples'], color=colors)
    ax1.set_xlabel('Strategy #')
    ax1.set_ylabel('Sample Count')
    ax1.set_title('Sample Count by Strategy')
    ax1.set_xticks(valid_results['strategy'])
    for bar, count in zip(bars, valid_results['samples']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{count:,}', ha='center', va='bottom', fontsize=8)
    
    # Plot 2: Insurance distribution comparison
    ax2 = axes[0, 1]
    x = np.arange(len(valid_results))
    width = 0.35
    bars1 = ax2.bar(x - width/2, valid_results['private'], width, label='Private', color='#3498db')
    bars2 = ax2.bar(x + width/2, valid_results['medicaid_medicare'], width, label='Medicaid/Medicare', color='#2ecc71')
    ax2.set_xlabel('Strategy #')
    ax2.set_ylabel('Count')
    ax2.set_title('Insurance Distribution by Strategy')
    ax2.set_xticks(x)
    ax2.set_xticklabels(valid_results['strategy'])
    ax2.legend()
    
    # Plot 3: Private percentage
    ax3 = axes[1, 0]
    bars = ax3.bar(valid_results['strategy'], valid_results['private_pct'], color=colors)
    ax3.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% (balanced)')
    ax3.set_xlabel('Strategy #')
    ax3.set_ylabel('Private %')
    ax3.set_title('Private Insurance Percentage by Strategy')
    ax3.set_xticks(valid_results['strategy'])
    ax3.set_ylim(0, 100)
    ax3.legend()
    for bar, pct in zip(bars, valid_results['private_pct']):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
    
    # Plot 4: File sizes
    ax4 = axes[1, 1]
    bars = ax4.bar(valid_results['strategy'], valid_results['file_size_mb'], color=colors)
    ax4.set_xlabel('Strategy #')
    ax4.set_ylabel('File Size (MB)')
    ax4.set_title('File Size by Strategy')
    ax4.set_xticks(valid_results['strategy'])
    for bar, size in zip(bars, valid_results['file_size_mb']):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{size:.0f}', ha='center', va='bottom', fontsize=8)
    
    # Legend for colors
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#3498db', label='Random Strategies (1-5)'),
                      Patch(facecolor='#e74c3c', label='Coreset Strategies (6-11)')]
    fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=2)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(os.path.join(CONFIG['OUTPUT_DIR'], 'strategies_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nVisualization saved to: {os.path.join(CONFIG['OUTPUT_DIR'], 'strategies_comparison.png')}")
else:
    print("No valid results to visualize")

## 4. Detailed Insurance Analysis

In [ ]:
print("\n" + "=" * 80)
print("4. DETAILED INSURANCE ANALYSIS")
print("=" * 80)

if len(valid_results) > 0:
    print(f"\n{'Strategy':<5} {'Name':<30} {'Type':<10} {'Samples':<10} {'Private %':<12} {'Balance':<15}")
    print("-" * 95)
    
    for _, row in valid_results.iterrows():
        # Calculate balance
        minority = min(row['private'], row['medicaid_medicare'])
        majority = max(row['private'], row['medicaid_medicare'])
        balance_ratio = minority / majority if majority > 0 else 0
        
        if balance_ratio >= 0.8:
            balance_status = 'Well balanced'
        elif balance_ratio >= 0.5:
            balance_status = 'Slight imbalance'
        else:
            balance_status = 'Imbalanced'
        
        print(f"{row['strategy']:<5} {row['name']:<30} {row['type']:<10} {row['samples']:<10,} {row['private_pct']:<11.1f}% {balance_status:<15}")
    
    print("-" * 95)
    
    # Strategy type summary
    print(f"\n[STRATEGY TYPE SUMMARY]")
    random_strats = valid_results[valid_results['type'] == 'Random']
    coreset_strats = valid_results[valid_results['type'] == 'Coreset']
    
    if len(random_strats) > 0:
        print(f"\nRandom Strategies (1-5):")
        print(f"  Count: {len(random_strats)}")
        print(f"  Avg samples: {random_strats['samples'].mean():.0f}")
        print(f"  Avg Private %: {random_strats['private_pct'].mean():.1f}%")
    
    if len(coreset_strats) > 0:
        print(f"\nCoreset Strategies (6-11):")
        print(f"  Count: {len(coreset_strats)}")
        print(f"  Avg samples: {coreset_strats['samples'].mean():.0f}")
        print(f"  Avg Private %: {coreset_strats['private_pct'].mean():.1f}%")
        print(f"  Note: Coreset sizes vary based on pre-selection (agnostic to insurance)")
else:
    print("No valid results to analyze")

## 5. Save Report

In [ ]:
print("\n" + "=" * 80)
print("5. SAVING REPORT")
print("=" * 80)

# Save full report
report_path = os.path.join(CONFIG['OUTPUT_DIR'], 'strategies_validation_report.csv')
results_df.to_csv(report_path, index=False)
print(f"\nFull report saved to: {report_path}")

# Summary statistics
if len(valid_results) > 0:
    summary_stats = {
        'total_strategies_checked': len(results_df),
        'strategies_found': len(valid_results),
        'strategies_missing': len(results_df) - len(valid_results),
        'all_embeddings_valid': (valid_results['embedding_size'] == npz_flat_size).all(),
        'total_unique_samples': valid_results['samples'].sum(),
        'avg_private_pct': valid_results['private_pct'].mean(),
        'total_file_size_mb': valid_results['file_size_mb'].sum()
    }
    
    summary_df = pd.DataFrame([summary_stats])
    summary_path = os.path.join(CONFIG['OUTPUT_DIR'], 'strategies_summary.csv')
    summary_df.to_csv(summary_path, index=False)
    print(f"Summary saved to: {summary_path}")
    
    print(f"\n[SUMMARY STATS]")
    for key, value in summary_stats.items():
        print(f"  {key}: {value}")

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)

if len(valid_results) == 11:
    print("\n✓ All 11 strategies validated successfully!")
    print("\nRecommendations:")
    print("  - Strategies 1-5: Random sampling methods (~2,000 samples each)")
    print("  - Strategies 6-11: Coreset methods (~1,100-1,185 samples each)")
    print("  - All use same embedding format: (8, 8, 1376) = 88,064 features")
    print("  - Load with: df = pd.read_pickle('data_typeX_insurance.pkl')")
    print("  - Embeddings: X = np.stack([e.flatten() for e in df['embedding']])")
    print("  - Labels: y = df['new_insurance_type'].map({'Medicaid_Medicare': 0, 'Private': 1}).values")
elif len(valid_results) > 0:
    print(f"\n⚠️ Only {len(valid_results)}/11 strategies found")
    missing = set(range(1, 12)) - set(valid_results['strategy'])
    print(f"   Missing strategies: {sorted(missing)}")
    print(f"   Run get_coreset_data.ipynb to generate missing datasets")
else:
    print("\n✗ No strategy files found")
    print("   Run get_coreset_data.ipynb to generate datasets")

print("=" * 80)